# TikiTorch — tropical deep learning with a PyTorch-shaped workflow

**TikiTorch** is a PyTorch-shaped library for tropical, morphological, and
hard-routed learning. It includes a general host-side research API and a
narrower audited subset for concrete compare/add/shift recipes. A tropical
operator does not, by itself, make its loss, backward rule, optimizer, host
boundary, or compiled hardware multiplier-free.

In the tropical semiring, "multiply" becomes ordinary **addition** and "add"
becomes a **min** or **max** (a comparison). A linear layer
`y = Wx` becomes a *morphological* operation `y_j = min_i (W_ji + x_i)` (erosion)
or `max_i (W_ji + x_i)` (dilation). Stack those, add a tropical attention or a
morphological convolution, train with a sign-based optimizer, and you have a deep
net built around sparse, discrete routes.

This notebook is a 10-minute tour. The others go deeper:
- **02_tropical_math** — the semiring, dilation/erosion, the soft→hard curriculum.
- **03_vision_cnn** — a morphological CNN and its declared arithmetic boundaries.
- **04_sequence** — sequence modeling and alternative loss/optimizer boundaries.


In [ ]:
# --- Setup: install TikiTorch (run once) ---
# TikiTorch is research code, not on PyPI. In Colab, install from the repo.
try:
    import tikitorch  # already installed?
except ImportError:
    REPO_URL = "https://github.com/nyuolab/tikitorch.git"
    import subprocess, sys, os
    if not os.path.exists("tikitorch_repo"):
        subprocess.run(
            ["git", "clone", "--branch", "pc-morphssm", "--single-branch",
             REPO_URL, "tikitorch_repo"],
            check=True,
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", "tikitorch_repo"],
        check=True,
    )
    import tikitorch
import torch
print("TikiTorch", getattr(tikitorch, "__version__", "?"), "| torch", torch.__version__)


## A complete training step in ten lines

TikiTorch mirrors the PyTorch namespace: `tt.nn`, `tt.optim`, `tt.models`,
`tt.functional`. Here is a morphological state-space LM trained one step with a
shift-oriented optimizer. This is an API example, not a full-step closure claim.


In [ ]:
import torch
import tikitorch as tt
import tikitorch.optim as optim

model = tt.models.MorphSSM(d_model=128, n_layers=2, task_type="lm", vocab_size=128)
opt = optim.PureTropicalSGD(model.parameters(), lr_shift=10)  # lr = 2^-10, a bit-shift

x = torch.randint(0, 128, (8, 64))   # (batch, seq)
y = torch.randint(0, 128, (8, 64))
loss = model.compute_loss(x, y)      # cross-entropy
loss.backward()                       # gradients route through argmin/argmax
opt.step(); opt.zero_grad()
print("loss:", round(loss.item(), 4))


## Closure is checked at declared boundaries

`assert_closed` is a structural deny-list over registered modules. Runtime and
full-training-step auditors cover executed paths for representative inputs.
Passing a software audit is useful evidence, but it is not a proof about
unexecuted branches or a synthesized hardware build.


In [ ]:
from tikitorch.audit import assert_closed, find_closure_violations

cnn = tt.models.TropicalCNN(in_channels=1, num_classes=10)   # morphological CNN
assert_closed(cnn)
print("TropicalCNN passes the structural module audit (params:", cnn.param_count, ")")

# A standard nn.Linear head is a *declared* DSP boundary — the checker names it:
boundary = tt.models.TropicalCNN(head_type="linear")
print("violations in linear-head variant:", find_closure_violations(boundary))


## Why this matters

Hard tropical routes can be implemented with comparisons, additions, shifts,
and data movement. TikiTorch includes HLS and AWS F2 research artifacts for
specific multiplier-free inference and learning kernels. Those physical claims
belong to the exact built configurations; the software model in this notebook
does not inherit them automatically. See the repository claim ledger for the
measured silicon results and their limitations.

**The map:**

| namespace | what |
|---|---|
| `tt.functional` | stateless tropical ops + autograd (linear, dilation, conv2d, attention, losses) |
| `tt.nn` | layer modules (TropicalLinear, TropicalConv2d, TropicalAttention, TropicalLayerNorm, ...) |
| `tt.optim` | sign-, shift-, and counter-oriented optimizer variants |
| `tt.models` | ready architectures (TropicalMLP/CNN, MorphSSM, TropicalTransformer, MoTE) |
| `tt.audit` | structural, runtime, and full-step closure checks |
